# Ejemplos básicos para ir aprendiendo y poner de baseline

A partir de la información del libro `Trading Algorítmico con Python` de Isaac Trullàs

In [ ]:
from utils_features import *
import pandas as pd
import yfinance as yf
import talib as ta
import re
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score, accuracy_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans, DBSCAN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from arch import arch_model

import backtrader as bt

In [2]:
spec, df_dia, df_min = load_future('gc1', 'all')

# Análisis técnico

## Indicadores de tendencia

In [3]:
# Media movil simple
def sma(df, n=20):
    df[f"sma_{n}"] = df['close'].rolling(n).mean()
    return df
# Media movil exponencial
def ema(df, n=20):
    df[f"ema_{n}"] = df['close'].ewm(span=n, adjust=False).mean()
    return df

In [4]:
# Convergencia y divergencia de medias moviles
def macd(df, n_fast=12, n_slow=26):
    df['ema_fast'] = df['close'].ewm(span=n_fast, adjust=False).mean()
    df['ema_slow'] = df['close'].ewm(span=n_slow, adjust=False).mean()
    df['macd'] = df['ema_fast'] - df['ema_slow']
    df['signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['histogram'] = df['macd'] - df['signal']
    return df

In [5]:
# Índice direccional medio
def adx(df, n=14):
    df['adx'] = ta.ADX(df['high'], df['low'], df['close'], timeperiod=n)
    return df

## Indicadores de momento

In [6]:
# Índice de fuerza relativa
def rsi(df, n=14):
    df['rsi'] = ta.RSI(df['close'], timeperiod=n)
    return df

In [7]:
# Estocástico
def stoch(df, n=14, slowk_period=3, slowd_period=3, slowk_matype=0, slowd_matype=0):
    df['slowk'], df['slowd'] = ta.STOCH(df['high'], df['low'], df['close'], fastk_period=n, slowk_period=slowk_period, slowd_period=slowd_period, slowk_matype=slowk_matype, slowd_matype=slowd_matype)
    return df

In [8]:
# Índice de fuerza de elder
def elder_force_index(df, n=13):
    df['efi'] = ta.EMA((df['close'] - df['close'].shift(1)) * df['volume'], timeperiod=n)
    return df

## Indicadores de volatilidad

In [9]:
# Bandas de bollinger
def bollinger_bands(df, n=20, num_std_dev=2):
    df['bb_middle'] = df['close'].rolling(n).mean()
    df['bb_std'] = df['close'].rolling(n).std()
    df['bb_upper'] = df['bb_middle'] + num_std_dev * df['bb_std']
    df['bb_lower'] = df['bb_middle'] - num_std_dev * df['bb_std']
    return df
# Repensar la función anterior

In [10]:
# Rango verdadero medio
def true_range(df):
    df['atr'] = ta.TRANGE(df['high'], df['low'], df['close'])
    return df
# Repensar la función anterior

In [11]:
# Commodity Channel Index
def cci(df, n=20):
    df['cci'] = ta.CCI(df['high'], df['low'], df['close'], timeperiod=n)
    return df

## Indicadores de volumen

In [12]:
# Volumen en balance
def obv(df):
    df['obv'] = ta.OBV(df['close'], df['volume'])
    return df
# Repensar la función anterior

In [13]:
# Volumen relativo
def vpt(df):
    df['vpt'] = (df['close'] - df['close'].shift(1)) / df['close'].shift(1) * df['volume']
    return df
# Repensar la función anterior

In [14]:
# Acumulación/distribución
def ad(df):
    df['ad'] = ta.AD(df['high'], df['low'], df['close'], df['volume'])
    return df
# Repensar la función anterior

In [15]:
def generate_features(df, config_json=None):
    """Genera indicadores técnicos en el DataFrame `df`.
    `config_json` puede ser un diccionario o un string JSON con configuraciones por indicador,
    por ejemplo: {'sma': {'n':20}, 'macd': {'n_fast':12,'n_slow':26}}
    Devuelve el DataFrame con nuevas columnas."""
    import json
    cfg = {}
    if config_json:
        if isinstance(config_json, str):
            try:
                cfg = json.loads(config_json)
            except Exception as e:
                raise ValueError('config_json string no es JSON válido: ' + str(e))
        elif isinstance(config_json, dict):
            cfg = config_json
        else:
            raise ValueError('config_json debe ser dict o JSON string')
    # helper para obtener parámetros (devuelve dict vacío si no hay configuración)
    def p(name):
        return cfg.get(name, {}) if cfg else {}
    # función auxiliar para llamadas seguras donde se pasan kwargs sólo si existen
    def _call(func, name, pass_cfg=True):
        params = p(name)
        if pass_cfg and params:
            return func(df, **params)
        return func(df)
    # Aplicar indicadores (pasar kwargs sólo cuando la función los acepta)
    df = df.copy()
    df = _call(sma, 'sma')
    df = _call(ema, 'ema')
    df = _call(macd, 'macd')
    df = _call(adx, 'adx')
    df = _call(rsi, 'rsi')
    df = _call(stoch, 'stoch')
    df = _call(elder_force_index, 'elder_force_index')
    df = _call(bollinger_bands, 'bollinger_bands')
    # true_range en este notebook no recibe kwargs, llamarla directamente
    df = true_range(df)
    df = _call(cci, 'cci')
    df = _call(obv, 'obv', pass_cfg=False)
    df = _call(vpt, 'vpt', pass_cfg=False)
    df = _call(ad, 'ad', pass_cfg=False)
    return df

In [16]:
df_dia_features = generate_features(df_dia)
df_min_features = generate_features(df_min)

In [17]:
def resample_ohlcv(df, period="5min"):
    """
    Resamplea un dataframe OHLCV al periodo deseado.
    period puede ser: '1min', '5min', '15min', '30min', '1H', '1D', etc.
    """

    # Asegurar orden temporal
    df = df.sort_values("datetime").copy()

    # Asegurar que datetime es datetime64
    df["datetime"] = pd.to_datetime(df["datetime"])

    # Establecer índice temporal
    df = df.set_index("datetime")

    # Diccionario OHLCV estándar
    ohlc_dict = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
        "openint": "last"
    }

    # Resample usando el periodo elegido
    df_resampled = df.resample(period).agg(ohlc_dict)

    # Eliminar velas vacías
    df_resampled = df_resampled.dropna(subset=["open", "high", "low", "close"])

    # Añadir columnas extra
    df_resampled["ticker"] = df["ticker"].iloc[0]
    df_resampled["per"] = period

    # Reset index
    df_resampled = df_resampled.reset_index()

    return df_resampled


In [18]:
df_5_min = resample_ohlcv(df_min)

In [19]:
df_5_min_features = generate_features(df_5_min)

In [20]:
# Generar datos ohlcv acumulados diarios a partir de los datos de 5 minutos
def daily_ohlcv_cummulative(df_5_min):
    df = df_5_min.copy()
    
    # Asegurar orden temporal
    df = df.sort_values('datetime')
    
    # Crear columna de día
    df['date'] = df['datetime'].dt.date
    
    # Open del día (primer valor de cada grupo)
    df['open_day'] = df.groupby('date')['open'].transform('first')
    
    # High acumulado intradía
    df['high_cum'] = df.groupby('date')['high'].cummax()
    
    # Low acumulado intradía
    df['low_cum'] = df.groupby('date')['low'].cummin()
    
    # Volume acumulado intradía
    df['volume_cum'] = df.groupby('date')['volume'].cumsum()
    
    # Open interest (último valor hasta ese momento → ya es el actual)
    df['openint_cum'] = df['openint']
    
    # (Opcional) puedes sobrescribir columnas originales
    # df['open'] = df['open_day']
    # df['high'] = df['high_cum']
    # df['low'] = df['low_cum']
    # df['volume'] = df['volume_cum']
    
    # Limpiar columnas auxiliares si quieres
    # df = df.drop(columns=['date', 'open_day', 'high_cum', 'low_cum', 'volume_cum'])
    
    return df

In [21]:
df_dia_cum = daily_ohlcv_cummulative(df_5_min)

In [22]:
df_dia_cum_features = generate_features(df_dia_cum)

In [23]:
# swing_dia = calculate_swings(df_dia, strategy='atr', atr_period=14, atr_mult=1.5)

In [24]:
df_base = pd.concat([df_5_min_features])

In [25]:
df_5_min

,datetime,open,high,low,close,volume,openint,ticker,per
0,2010-09-20 07:00:00,1279.8,1280.7,1279.8,1279.8,72,0.0,GC,5min
1,2010-09-20 07:05:00,1279.9,1280.1,1279.8,1279.8,25,0.0,GC,5min
2,2010-09-20 07:10:00,1279.8,1280.1,1279.8,1280.1,26,0.0,GC,5min
3,2010-09-20 07:15:00,1280.1,1280.6,1280.1,1280.5,35,0.0,GC,5min
4,2010-09-20 07:20:00,1280.5,1280.7,1280.4,1280.6,26,0.0,GC,5min
...,...,...,...,...,...,...,...,...,...
925344,2023-09-19 06:40:00,1953.8,1953.8,1953.4,1953.5,129,0.0,GC,5min
925345,2023-09-19 06:45:00,1953.5,1953.8,1953.5,1953.7,63,0.0,GC,5min
925346,2023-09-19 06:50:00,1953.7,1953.8,1953.7,1953.8,25,0.0,GC,5min
925347,2023-09-19 06:55:00,1953.8,1953.8,1953.5,1953.6,145,0.0,GC,5min


In [26]:
df_dia_cum

,datetime,open,high,low,close,volume,openint,ticker,per,date,open_day,high_cum,low_cum,volume_cum,openint_cum
0,2010-09-20 07:00:00,1279.8,1280.7,1279.8,1279.8,72,0.0,GC,5min,2010-09-20,1279.8,1280.7,1279.8,72,0.0
1,2010-09-20 07:05:00,1279.9,1280.1,1279.8,1279.8,25,0.0,GC,5min,2010-09-20,1279.8,1280.7,1279.8,97,0.0
2,2010-09-20 07:10:00,1279.8,1280.1,1279.8,1280.1,26,0.0,GC,5min,2010-09-20,1279.8,1280.7,1279.8,123,0.0
3,2010-09-20 07:15:00,1280.1,1280.6,1280.1,1280.5,35,0.0,GC,5min,2010-09-20,1279.8,1280.7,1279.8,158,0.0
4,2010-09-20 07:20:00,1280.5,1280.7,1280.4,1280.6,26,0.0,GC,5min,2010-09-20,1279.8,1280.7,1279.8,184,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
925344,2023-09-19 06:40:00,1953.8,1953.8,1953.4,1953.5,129,0.0,GC,5min,2023-09-19,1955.2,1956.0,1953.2,8073,0.0
925345,2023-09-19 06:45:00,1953.5,1953.8,1953.5,1953.7,63,0.0,GC,5min,2023-09-19,1955.2,1956.0,1953.2,8136,0.0
925346,2023-09-19 06:50:00,1953.7,1953.8,1953.7,1953.8,25,0.0,GC,5min,2023-09-19,1955.2,1956.0,1953.2,8161,0.0
925347,2023-09-19 06:55:00,1953.8,1953.8,1953.5,1953.6,145,0.0,GC,5min,2023-09-19,1955.2,1956.0,1953.2,8306,0.0


In [27]:
df_5_min_features

,datetime,open,high,low,close,volume,openint,ticker,per,sma_20,...,efi,bb_middle,bb_std,bb_upper,bb_lower,atr,cci,obv,vpt,ad
0,2010-09-20 07:00:00,1279.8,1280.7,1279.8,1279.8,72,0.0,GC,5min,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,NaN,-7.200000e+01
1,2010-09-20 07:05:00,1279.9,1280.1,1279.8,1279.8,25,0.0,GC,5min,NaN,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,72.0,0.000000,-9.700000e+01
2,2010-09-20 07:10:00,1279.8,1280.1,1279.8,1280.1,26,0.0,GC,5min,NaN,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,98.0,0.006095,-7.100000e+01
3,2010-09-20 07:15:00,1280.1,1280.6,1280.1,1280.5,35,0.0,GC,5min,NaN,...,NaN,NaN,NaN,NaN,NaN,0.5,NaN,133.0,0.010937,-5.000000e+01
4,2010-09-20 07:20:00,1280.5,1280.7,1280.4,1280.6,26,0.0,GC,5min,NaN,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,159.0,0.002030,-4.133333e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
925344,2023-09-19 06:40:00,1953.8,1953.8,1953.4,1953.5,129,0.0,GC,5min,1954.100,...,-11.318216,1954.100,0.337171,1954.774342,1953.425658,0.4,-141.283215,190755.0,-0.019808,5.170126e+06
925345,2023-09-19 06:45:00,1953.5,1953.8,1953.5,1953.7,63,0.0,GC,5min,1954.075,...,-7.901328,1954.075,0.347737,1954.770474,1953.379526,0.3,-101.204819,190818.0,0.006450,5.170147e+06
925346,2023-09-19 06:50:00,1953.7,1953.8,1953.7,1953.8,25,0.0,GC,5min,1954.055,...,-6.415424,1954.055,0.351650,1954.758300,1953.351700,0.1,-69.767442,190843.0,0.001280,5.170172e+06
925347,2023-09-19 06:55:00,1953.8,1953.8,1953.5,1953.6,145,0.0,GC,5min,1954.005,...,-9.641792,1954.005,0.341012,1954.687025,1953.322975,0.3,-92.977250,190698.0,-0.014843,5.170123e+06


In [28]:
df_dia_cum_features

,datetime,open,high,low,close,volume,openint,ticker,per,date,...,efi,bb_middle,bb_std,bb_upper,bb_lower,atr,cci,obv,vpt,ad
0,2010-09-20 07:00:00,1279.8,1280.7,1279.8,1279.8,72,0.0,GC,5min,2010-09-20,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,NaN,-7.200000e+01
1,2010-09-20 07:05:00,1279.9,1280.1,1279.8,1279.8,25,0.0,GC,5min,2010-09-20,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,72.0,0.000000,-9.700000e+01
2,2010-09-20 07:10:00,1279.8,1280.1,1279.8,1280.1,26,0.0,GC,5min,2010-09-20,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,98.0,0.006095,-7.100000e+01
3,2010-09-20 07:15:00,1280.1,1280.6,1280.1,1280.5,35,0.0,GC,5min,2010-09-20,...,NaN,NaN,NaN,NaN,NaN,0.5,NaN,133.0,0.010937,-5.000000e+01
4,2010-09-20 07:20:00,1280.5,1280.7,1280.4,1280.6,26,0.0,GC,5min,2010-09-20,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,159.0,0.002030,-4.133333e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
925344,2023-09-19 06:40:00,1953.8,1953.8,1953.4,1953.5,129,0.0,GC,5min,2023-09-19,...,-11.318216,1954.100,0.337171,1954.774342,1953.425658,0.4,-141.283215,190755.0,-0.019808,5.170126e+06
925345,2023-09-19 06:45:00,1953.5,1953.8,1953.5,1953.7,63,0.0,GC,5min,2023-09-19,...,-7.901328,1954.075,0.347737,1954.770474,1953.379526,0.3,-101.204819,190818.0,0.006450,5.170147e+06
925346,2023-09-19 06:50:00,1953.7,1953.8,1953.7,1953.8,25,0.0,GC,5min,2023-09-19,...,-6.415424,1954.055,0.351650,1954.758300,1953.351700,0.1,-69.767442,190843.0,0.001280,5.170172e+06
925347,2023-09-19 06:55:00,1953.8,1953.8,1953.5,1953.6,145,0.0,GC,5min,2023-09-19,...,-9.641792,1954.005,0.341012,1954.687025,1953.322975,0.3,-92.977250,190698.0,-0.014843,5.170123e+06


In [ ]:
df = df_dia_cum_features.join(df_5_min_features.set_index('datetime'), on='datetime', rsuffix='_5min')

Index(['datetime', 'open', 'high', 'low', 'close', 'volume', 'openint',
       'ticker', 'per', 'date', 'open_day', 'high_cum', 'low_cum',
       'volume_cum', 'openint_cum', 'sma_20', 'ema_20', 'ema_fast', 'ema_slow',
       'macd', 'signal', 'histogram', 'adx', 'rsi', 'slowk', 'slowd', 'efi',
       'bb_middle', 'bb_std', 'bb_upper', 'bb_lower', 'atr', 'cci', 'obv',
       'vpt', 'ad', 'open_5min', 'high_5min', 'low_5min', 'close_5min',
       'volume_5min', 'openint_5min', 'ticker_5min', 'per_5min', 'sma_20_5min',
       'ema_20_5min', 'ema_fast_5min', 'ema_slow_5min', 'macd_5min',
       'signal_5min', 'histogram_5min', 'adx_5min', 'rsi_5min', 'slowk_5min',
       'slowd_5min', 'efi_5min', 'bb_middle_5min', 'bb_std_5min',
       'bb_upper_5min', 'bb_lower_5min', 'atr_5min', 'cci_5min', 'obv_5min',
       'vpt_5min', 'ad_5min'],
      dtype='str')

# Modelos predictivos

## Regresión lineal y logística

In [ ]:
# Regresion lineal
X = df.drop(columns=["close"])
Y = df["close"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")



In [ ]:
# Regresion logistica
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
X = df.drop(columns=["close", "target"])
Y = df["target"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

preccision = accuracy_score(Y_test, Y_pred)
matriz_confusion = confusion_matrix(Y_test, Y_pred)
roc_auc = roc_auc_score(Y_test, model.predict(X_test)[:, 1])

print(f"Accuracy: {preccision}")
print(f"Confusion Matrix:\n{matriz_confusion}")
print(f"ROC AUC Score: {roc_auc}")

## Modelos de series temporales

In [ ]:
# ARIMA
# df.reset_index()
dates = df["date"]
close = df["close"]
close.index = range(len(close))
model = ARIMA(close, order=(5, 1, 2))
model_fit = model.fit()

print(model_fit.summary())

in_sample_pred = model_fit.predict(start=0, end=len(close)-1)

plt.figure(figsize=(12, 6))
plt.plot(dates, close, label='Actual')
plt.plot(dates, in_sample_pred, label='Predicted', alpha=0.7)
plt.title('ARIMA In-Sample Prediction') 
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

In [ ]:
# GARCH
df['returns'] = df['close'].pct_change().dropna()*100
model = arch_model(df['returns'].dropna(), vol='Garch', p=1, q=1)
model_fit = model.fit()

print(model_fit.summary())

in_sample_pred = model_fit.conditional_volatility

oos_pred = model_fit.forecast(horizon=10)
forecast_variance = oos_pred.variance.iloc[-1]
forecast_volatility_oos = np.sqrt(forecast_variance)

df["historical_volatility"] = df['returns'].rolling(window=20).std()

future_dates = pd.date_range(start=df['datetime'].iloc[-1] + pd.Timedelta(minutes=5), periods=10, freq='5min')
plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['historical_volatility'], label='Historical Volatility')
plt.plot(future_dates, forecast_volatility_oos.values, label='Forecasted Volatility', marker='o')

plt.plot(df['datetime'], in_sample_pred, label='In-Sample Volatility', alpha=0.7)
plt.title('GARCH Volatility Forecast')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.show()

In [ ]:
# SARIMA
# df.reset_index()
dates = df["date"]
close = df["close"]
close.index = range(len(close))
model = SARIMAX(close, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
model_fit = model.fit()

print(model_fit.summary())

in_sample_pred = model_fit.predict(start=0, end=len(close)-1)

plt.figure(figsize=(12, 6))
plt.plot(dates, close, label='Actual')
plt.plot(dates, in_sample_pred, label='Predicted', alpha=0.7)
plt.title('SARIMA In-Sample Prediction')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

## Machine learning

In [ ]:
# Decision Tree Classifier
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
X = df.drop(columns=["close", "target"])
Y = df["target"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier()
model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

accuracy = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{classification_report(Y_test, Y_pred)}")

In [ ]:
# Random forests
# df["SMA_10"]
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
X = df.drop(columns=["close", "target"])
Y = df["target"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model = RandomForestClassifier()
grid_search = GridSearchCV(estimator=model, param_grid={
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}, cv=3, n_jobs=-1, verbose=2)

grid_search.fit(X_train, Y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")

best_rf = grid_search.best_estimator_
Y_pred = best_rf.predict(X_test)
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{classification_report(Y_test, Y_pred)}")

In [ ]:
# LSTM
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(df.drop(columns=["close"]))
X_train = []
Y_train = []
window_size = 60
for i in range(window_size, len(X_scaled)):
    X_train.append(X_scaled[i-window_size:i, 0])
    Y_train.append(X_scaled[i, 0])
    
X_train, Y_train = np.array(X_train), np.array(Y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

model = Sequential()

model.add(LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], 1)))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=1))

model.compile(optimizer='adam', loss='mean_squared_error')

model.fit(X_train, Y_train, epochs=50, batch_size=32)

df_test = df[-window_size:].copy()
df_total = pd.concat([df, df_test], axis=0)
inputs = df_total[len(df_total) - len(df_test) - window_size:]['close'].values
inputs = inputs.reshape(-1, 1)
inputs = scaler.transform(inputs)

X_test = []
for i in range(window_size, len(inputs)):
    X_test.append(inputs[i-window_size:i, 0])
    
X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

predicted_price = model.predict(X_test)
predicted_price = scaler.inverse_transform(predicted_price)

plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['close'], label='Actual Price')
plt.plot(df_test['datetime'], predicted_price, label='Predicted Price', alpha=0.7)
plt.title('LSTM Price Prediction')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

rmse = np.sqrt(mean_squared_error(df_test['close'], predicted_price))
mae = mean_absolute_error(df_test['close'], predicted_price)
mape = mean_absolute_percentage_error(df_test['close'], predicted_price)

print(f"RMSE: {rmse}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")

In [ ]:
# Clustering
# Kmeans
X = df.drop(columns=["close"])
kmeans = KMeans(n_clusters=4, random_state=42)
df['cluster'] = kmeans.fit_predict(X)

sns.scatterplot(data=df, x='rsi', y='adx', hue='cluster', palette='Set1')
plt.title('Clustering de Indicadores Técnicos')
plt.xlabel('RSI')
plt.ylabel('ADX')
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df['datetime'], df['close'], label='Close Price')
for cluster in df['cluster'].unique():
    cluster_data = df[df['cluster'] == cluster]
    plt.scatter(cluster_data['datetime'], cluster_data['close'], label=f'Cluster {cluster}', alpha=0.6)
plt.title('Price Movement by Cluster')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

In [ ]:
# DBSCAN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=["close"]))

dbscan = DBSCAN(eps=0.5, min_samples=5)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

plt.figure(figsize=(12, 6))
noise = df[df['dbscan_cluster'] == -1]
clusters = df[df['dbscan_cluster'] != -1]

plt.scatter(noise['datetime'], noise['close'], label='Noise', color='red', alpha=0.6)
plt.scatter(clusters['datetime'], clusters['close'], c=clusters['dbscan_cluster'], cmap='Set1', label='Clusters', alpha=0.6)
plt.title('DBSCAN Clustering of Price Movements')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()

# Backtesting

# Evaluación de resultados

In [ ]:
# Rendimiento total y anualizado

In [ ]:
# Ratio Sharpe


In [49]:
# Máximo drawdown


In [50]:
# Ratio Profit and Loss


In [51]:
# Costes operativos
